# Benchmark de la GPU de Colab
Primero prepare el código y Drive con las celdas 0–3 de `colab_training.ipynb`. Este notebook solo llama al script y muestra las mediciones. No instala paquetes. Use `SMOKE=True` para el modo pequeño en CPU, únicamente en un entorno de pruebas autorizado.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
import pandas as pd

REPO_DIR = Path("/content/quant-ai")
SMOKE = False
# Ingrese cantidades reales aproximadas; vacío desactiva la estimación de épocas.
ROWS_PER_SYMBOL = {}
N_FEATURES = 64  # Ajustar al número de características del dataset.
if not (REPO_DIR / "scripts/gpu_benchmark.py").is_file():
    raise FileNotFoundError("Prepare primero /content/quant-ai con colab_training.ipynb.")
os.environ.setdefault("QUANT_AI_HOME", "/content/drive/MyDrive/quant-ai")
command = [sys.executable, str(REPO_DIR / "scripts/gpu_benchmark.py"),
           "--config", str(REPO_DIR / "configs/base.yaml"), "--n-features", str(N_FEATURES)]
if ROWS_PER_SYMBOL:
    command += ["--rows-per-symbol", *[f"{s}={n}" for s, n in ROWS_PER_SYMBOL.items()]]
if SMOKE:
    command.append("--smoke")
subprocess.run(command, cwd=REPO_DIR, check=True)

In [ ]:
from quant_ai.config import load_config
cfg = load_config(REPO_DIR / "configs/base.yaml")
path = max((cfg.reports_dir / "gpu_benchmark").glob("*.json"), key=lambda p: p.stat().st_mtime)
report = json.loads(path.read_text())
print(path)
print(report["estimate_scope"])
display(pd.DataFrame(report["matmul"]))
display(pd.json_normalize(report["training"]))

TFLOPS se calcula con `2 × N³ × iteraciones / tiempo / 10¹²`. El throughput mide pasos de optimización con datos sintéticos ya residentes en el dispositivo. La estimación no incluye validación, lectura desde Drive, purga, huecos ni escritura de checkpoints. `oom` indica que ese caso no cabe; `unsupported` indica que el formato no está disponible. Ninguno de esos casos recibe una cifra inventada.